# LAi Training on Google Colab
Train the LAi bilingual (HU/EN) model on a free T4 GPU.

**Steps:**
1. Clone repo & install dependencies
2. Prepare training data (chat-formatted HU/EN)
3. Build BPE vocabulary
4. Train model on GPU
5. Download trained model

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# 0. Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 1. Clone repo & install deps
!git clone https://github.com/ML6200/LAi2.git
%cd LAi2
!pip install -q psutil datasets tqdm

In [ ]:
# 2. Prepare training data
# Use 'tiny' size for more data (5K translations, 5K conversations, 3K HU, 3K stories)
# Use 'small' for even more if you have time
DATASET_SIZE = "tiny"  # @param ["micro", "tiny", "small"]

!python training/data.py --output data/train.txt --size {DATASET_SIZE}

In [ ]:
# 3. Build BPE vocabulary
VOCAB_SIZE = 16000  # @param {type:"integer"}

!python training/build_vocab.py \
    --data data/train.txt \
    --vocab_size {VOCAB_SIZE} \
    --output data/vocab.bin

In [ ]:
# 4. Train!
# Model configs:
#   micro: 10M params, 288 dim, 6 layers  (fastest, ~5 min on T4)
#   tiny:  ~30M params, 384 dim, 8 layers  (good balance, ~15 min on T4)
#   mini:  ~80M params, 512 dim, 12 layers (best quality, ~45 min on T4)

MODEL_CONFIG = "tiny"  # @param ["micro", "tiny", "mini"]
EPOCHS = 15  # @param {type:"integer"}
BATCH_SIZE = 64  # @param {type:"integer"}
GRAD_ACCUM = 4  # @param {type:"integer"}
LEARNING_RATE = 3e-4  # @param {type:"number"}

!python training/train.py \
    --config {MODEL_CONFIG} \
    --data data/train.txt \
    --vocab data/vocab.bin \
    --output models/lai-{MODEL_CONFIG}.bin \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --grad_accum {GRAD_ACCUM} \
    --lr {LEARNING_RATE} \
    --device cuda

In [ ]:
# 5. Quick test: generate text with the trained model (in Python)
import sys
sys.path.insert(0, 'training')
from train import Transformer, ModelConfig, BPETokenizer
import torch

# Load model
configs = {"micro": ModelConfig.micro, "tiny": ModelConfig.tiny, "mini": ModelConfig.mini}
config = configs[MODEL_CONFIG]()

tokenizer = BPETokenizer()
tokenizer.load('data/vocab.bin')
config.vocab_size = len(tokenizer.vocab)

model = Transformer(config).cuda()

# Load weights from the exported binary (skip header + vocab)
import struct
with open(f'models/lai-{MODEL_CONFIG}.bin', 'rb') as f:
    # Read header
    f.seek(0)
    magic = f.read(4)
    version = struct.unpack('I', f.read(4))[0]
    # Skip config fields
    for _ in range(8): f.read(4)  # 8 config fields
    f.read(2)  # 2 floats
    f.read(1)  # dtype
    vocab_offset = struct.unpack('Q', f.read(8))[0]
    weights_offset = struct.unpack('Q', f.read(8))[0]
    print(f"Weights at offset: {weights_offset}")

print("\n--- Quick generation test ---")
print("(For proper inference, use the C++ binary on your Mac)")
print(f"Model: {MODEL_CONFIG}, {model.num_params()/1e6:.1f}M params")

In [ ]:
# 6. Download the trained model
from google.colab import files

model_path = f'models/lai-{MODEL_CONFIG}.bin'
import os
size_mb = os.path.getsize(model_path) / 1e6
print(f"Downloading {model_path} ({size_mb:.1f} MB)...")
files.download(model_path)

## After downloading

Copy the model to your LAi project and test:

```bash
# On your Mac
mv ~/Downloads/lai-tiny.bin models/
make clean && make release
./lai -m models/lai-tiny.bin -p "Szia, hogy vagy?"
```

### Tips for better results
- Use `small` dataset size + `tiny` or `mini` model config
- Train for 20-30 epochs (watch the loss — stop when it plateaus)
- Final loss should be < 2.0 for coherent output, < 1.0 for good quality
- If loss is stuck > 4.0, increase dataset size